**Cálculo Numérico y Programación**

**Aux. Daniel Roberto Garcia Miranda**

**Matrices y sistemas lineales**

**Clasificación de los Métodos**

**A. Métodos Directos**

Calculan la solución exacta en un número finito de operaciones. Son ideales para matrices densas de tamaño pequeño o mediano.
* **Eliminación de Gauss:** Transformación a una matriz triangular superior.
* **Gauss-Jordan:** Transformación a la matriz identidad (reducción completa).
* **Factorización LU:** Descomposición en matrices triangular inferior ($L$) y superior ($U$).
* **Factorización de Cholesky:** Descomposición $LL^T$ para matrices simétricas y definidas positivas.

**B. Métodos Iterativos**

Generan una secuencia de aproximaciones que convergen a la solución. Se prefieren para matrices grandes y dispersas.
* **Método de Jacobi:** Basado en la separación de la diagonal.
* **Método de Gauss-Seidel:** Versión optimizada de Jacobi con actualización inmediata.
* **Método de Sobre-Relajación (SOR):** Aceleración de la convergencia mediante un parámetro $\omega$.

**Factorización LU**

In [ ]:
##Factorizacion LU

import numpy as np

M = [[2, 1, -1, 8],
     [-3, -1, 2, -11],
     [-2, 1, 2, -3]]

# A = [[2, 1, -1],
#      [-3, -1, 2],
#      [-2, 1, 2]]
# B = [8, -11, -3]

# con numpy
MM = np.array(M, dtype=float)
AA = MM[:, :-1].copy()
BB = MM[:, -1].copy()
n = len(M)

U = AA.copy()
L = np.eye(n,n)
print(L, U)
#Aca operamos y tenemos ambas matrices L y U
for i in range(n):
    for j in range(i+1,n):
        factor = U[j,i]/U[i,i]
        U[j] = U[j] - factor*U[i]
        L[j,i] = factor
print(L)
print(U)

#Ahora calculamos segun LUx = B ---> Ux = y, Ly = B
#Calculo de y

y = np.zeros(n)
for i in range(0,n):
    suma = 0
    for j in range(0,i):
        suma += L[i,j]*y[j]
    y[i] = (BB[i]- suma)/L[i,i]
print(y)

x = np.zeros(n)

for i in range(n-1,-1,-1):
    suma = 0 
    for j in range(i+1, n):
        suma += U[i,j]*x[j]
    x[i] = (y[i]-suma)/(U[i,i])

print(x)

**Gauss Seidel**

Método iterativo para resolver sistemas de ecuaciones de la forma Ax = B.

La formula general para las soluciones es:

$$x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j=1}^{i-1} a_{ij}x_j^{(k+1)} - \sum_{j=i+1}^{n} a_{ij}x_j^{(k)} \right)$$

No todas los sistemas se pueden resolver con gauss seidel, se exige que la matriz A sea diagonalmente dominante, es decir:

$$|a_{ii}| > \sum_{j \neq i} |a_{ij}|$$

In [ ]:
###Metodo Gauss Seidel
##Garcia Miranda Daniel Roberto

A = [[3.0, -0.1, -0.2],
     [0.1, 7.0, -0.3],
     [0.3, -0.2, 10.0]]
B = [7.85, -19.3, 71.4]


def GaussSeidel(A,B,iter):
    n = len(A)
    x = [0]*n
    for k in range(iter):
        # hist = list(x)
        for i in range(n):
            suma = B[i] 
            for j in range(n):
                if i!=j:
                    suma -=A[i][j]*x[j]
            x[i]=suma/A[i][i]
            print(x)
        
        # print(k, x)
    return x

xd = GaussSeidel(A,B,50)


In [ ]:
###Metodo Gauss Seidel con grafico
##Garcia Miranda Daniel Roberto

A = [[3.0, -0.1, -0.2],
     [0.1, 7.0, -0.3],
     [0.3, -0.2, 10.0]]
B = [7.85, -19.3, 71.4]

import matplotlib.pyplot as plt
import numpy as np
def GaussSeidelGRAF(A,B,iter):
    n = len(A)
    #La matriz para graficos
    matriz = []
    x = [0]*n
    for k in range(iter):
        hist = list(x)
        for i in range(n):
            suma = B[i] 
            for j in range(n):
                if i!=j:
                    suma -=A[i][j]*x[j]
            x[i]=suma/A[i][i]
        matriz.append(list(x))
        print(k, x)
    matriz = np.array(matriz)    
    iteracion = np.arange(len(matriz))
    for i in range(n):
        plt.scatter(iteracion, matriz[:, i],label=f'x_{i}')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xlabel('Número de Iteración')
    plt.ylabel('Valor de la incógnita')
    plt.title('Convergencia del Método de Gauss-Seidel')
    plt.show()
    return x

xd = GaussSeidelGRAF(A,B,50)


In [ ]:
A = [[1.0, 0.4, 0.4],
     [0.4, 1.0, 0.4],
     [0.4, 0.4, 1.0]]
B = [1.8, 1.8, 1.8]

GaussSeidelGRAF(A,B, 10)
A = [[1.0, 0.9],
     [0.9, 1.0]]
B = [1.9, 1.9]
GaussSeidelGRAF(A,B, 20)

**Metodo de Sobre Relajacion**

La lógica del "Peso" ($\omega$)En cada iteración, en lugar de tomar el valor nuevo de $x_i$ tal cual lo calcula Gauss-Seidel, tomamos una media ponderada entre el valor anterior y el nuevo:$$x_i^{(k+1)} = (1 - \omega)x_i^{(k)} + \omega x_i^{GS}$$

**El papel de $\omega$**

El valor de $\omega$ determina el comportamiento del método:

$\omega = 1$: El método se convierte exactamente en Gauss-Seidel.

$1 < \omega < 2$ (Sobrerrelajación): Se usa para acelerar la convergencia cuando el método es lento.

$0 < \omega < 1$ (Subrelajación): Se usa para estabilizar sistemas que de otra forma divergirían (es decir, cuando los valores "saltan" demasiado y nunca se asientan).

$\omega \geq 2$ o $\omega \leq 0$: El método generalmente diverge (no sirve).

In [ ]:
##Metodo de sobre relajacion
#Daniel Roberto Garcia Miranda

A = [[3.0, -0.1, -0.2],
     [0.1, 7.0, -0.3],
     [0.3, -0.2, 10.0]]
B = [7.85, -19.3, 71.4]

def SOR(A,B,iter, w):
    n = len(A)
    x = [0]*n
    for k in range(iter):
        hist = list(x)
        for i in range(n):
            suma = B[i] 
            for j in range(n):
                if i!=j:
                    suma -=A[i][j]*x[j]
            x_gs=suma/A[i][i]
            x[i] = (1-w)*x[i]+w*x_gs
        print(k, x)
    return x
# gg = GaussSeidel(A,B , 50)
xd =  SOR(A,B,50,3)
# xd =  SOR(A,B,50,1.)

![](Ejercicio1.png)
![](Ejercicio2.png)
![](Resnick.png)

![](MIT.png)


In [ ]:
#Ejemplo
MM = [[1, 10, 1, 0, 0, 0, 10],
      [2, 0, 20, 1, 0, 0, 10],
      [0, 3, 0, 0, 30, 3, 0],
      [10, 1, 0, 0, 0, -1, 5],
      [0, 0, 0, 2, -2, 20, 5],
      [0, 0, 1, 10, -1, 0, 0]]

def gauss_jordan(M):
    n = len(M)  ## numero de filas, ecuaciones
    m = len(M[0])  ## numero de columnas, variables + termino independiente
    for k in range(n):
        pivote = M[k][k]
        for j in range(m):
            M[k][j] /= pivote ##Normalizamos el pivote
        for i in range(n):
            if i != k: 
                factor = M[i][k]
                for j in range(m):
                    M[i][j] -= factor * M[k][j] #Restamos a las demás lineas

    XD = [pacman[-1] for pacman in M] # extraemos el vector solución
    return XD
M = [[2, 1, -1, 8],
     [-3, -1, 2, -11],
     [-2, 1, 2, -3]]
gauss_jordan(MM)

In [ ]:
A = [
     [10, 1, 0, 0, 0, -1],
     [1, 10, 1, 0, 0, 0],
     [2, 0, 20, 1, 0, 0],
     [0, 0, 1, 10, -1, 0],
     [0, 3, 0, 0, 30, 3],
     [0, 0, 0, 2, -2, 20]
]
B = [10, 10, 0, 5, 5, 0]
GaussSeidelGRAF(A, B, 30)